# 🎙️ Matraca Studio - Dublador & Clonador de Voz com IA
**Clone sua voz e duble qualquer vídeo ou áudio para múltiplos idiomas mantendo sincronia temporal!**

Desenvolvido para execução otimizada no Google Colab com GPU (T4, L4, A100).
- **IA de Clonagem Vocal:** [k2-fsa/OmniVoice](https://github.com/k2-fsa/OmniVoice)
- **IA de Reconhecimento de Fala & Timestamps:** OpenAI Whisper
- **Processamento de Áudio & Vídeo:** FFmpeg + Torchaudio
- **Interface:** Gradio com fila (Queue) para estabilidade total sem erros 500


In [ ]:
# @title Passo 1: Instalar Dependências e FFmpeg
# @markdown Instala OmniVoice, Whisper, Gradio, Deep-Translator e ferramentas de mídia.

!apt-get -y update -qq && apt-get -y install -qq ffmpeg
!pip install -q omnivoice gradio openai-whisper deep-translator
print('✅ Dependências e ferramentas de mídia instaladas com sucesso!')


In [ ]:
# @title Passo 2: Carregar os Modelos de IA na GPU (T4 / A100 / L4)
# @markdown Baixa e carrega o Whisper e o OmniVoice na memória de vídeo da GPU.

import os
import torch
import torchaudio
import whisper
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'🚀 Dispositivo em uso: {device}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🔥 GPU Detectada: {gpu_name} ({vram:.1f} GB VRAM)')
else:
    print('⚠️ GPU não detectada! Por favor, ative a GPU T4 no menu do Colab (Ambiente de Execução > Alterar tipo de ambiente de execução).')

print('\n📥 Carregando modelo Whisper (base) para transcrição de áudio...')
whisper_model = whisper.load_model('base', device=device)
print('✅ Whisper pronto!')

print('\n📥 Carregando OmniVoice da k2-fsa...')
omnivoice_model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('✅ OmniVoice carregado e pronto para clonar sua voz!')


In [ ]:
# @title Passo 3: Motor de Processamento, Sincronização Temporal e Remuxing de Vídeo
# @markdown Funções auxiliares para extração de áudio, medição de duração, normalização, time stretch suave e remuxing.

import subprocess
import json
import tempfile
import os
import re
import shutil
import gc
import time
import torch
import torchaudio
from deep_translator import GoogleTranslator

def get_media_info(file_path):
    """Retorna a duração em segundos e se o arquivo contém faixa de vídeo."""
    if not file_path or not os.path.exists(file_path):
        return 0.0, False
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration:stream=codec_type',
        '-of', 'json', file_path
    ]
    try:
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        data = json.loads(res.stdout)
        duration = float(data.get('format', {}).get('duration', 0.0))
        streams = data.get('streams', [])
        has_video = any(s.get('codec_type') == 'video' for s in streams)
        return duration, has_video
    except Exception as e:
        print(f'Erro ao inspecionar mídia com ffprobe: {e}')
        return 0.0, False

def extract_audio_to_wav(media_path, output_wav):
    """Converte qualquer mídia (vídeo ou áudio) em WAV 24kHz mono puro para o OmniVoice."""
    cmd = [
        'ffmpeg', '-y', '-i', media_path,
        '-vn', '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def extract_audio_slice(input_wav, start_sec, end_sec, output_slice_wav):
    """Extrai uma fatia pura de áudio com recorte exato."""
    duration = max(0.5, end_sec - start_sec)
    cmd = [
        'ffmpeg', '-y',
        '-ss', f'{start_sec:.3f}',
        '-t', f'{duration:.3f}',
        '-i', input_wav,
        '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_slice_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def normalize_audio_tensor(tensor, target_peak=0.95):
    """Normaliza o volume do tensor de áudio para evitar clipping ou volume muito baixo."""
    if tensor is None or tensor.numel() == 0:
        return tensor
    max_val = torch.max(torch.abs(tensor))
    if max_val > 1e-4:
        return tensor / max_val * target_peak
    return tensor

def select_best_voice_slice(segments, full_wav, orig_duration, output_slice_wav, whisper_model=None):
    """
    Identifica o trecho vocal mais limpo e representativo (4 a 8s),
    evitando músicas, vinhetas ou ruídos.
    Gera também o texto 100% alinhado com a fatia para o OmniVoice.
    """
    ref_slice_start = 0.0
    ref_slice_end = min(orig_duration, 6.0)

    # Filtrar segmentos com fala real (ignora ruídos e tags de música)
    valid_segments = []
    if segments:
        for s in segments:
            txt = s.get('text', '').strip()
            if not txt or re.search(r'(\[música\]|\[music\]|♪|♫|\(música\))', txt, re.IGNORECASE):
                continue
            if len(txt) >= 8 and (s['end'] - s['start']) >= 1.0:
                valid_segments.append(s)

    if valid_segments:
        best_combo = []
        for i in range(len(valid_segments)):
            combo = [valid_segments[i]]
            dur = combo[-1]['end'] - combo[0]['start']
            for j in range(i + 1, len(valid_segments)):
                if valid_segments[j]['start'] - valid_segments[j-1]['end'] < 1.5:
                    combo.append(valid_segments[j])
                    dur = combo[-1]['end'] - combo[0]['start']
                    if dur >= 5.0:
                        break
                else:
                    break
            if 4.0 <= dur <= 10.0:
                best_combo = combo
                break
            elif dur > (best_combo[-1]['end'] - best_combo[0]['start'] if best_combo else 0):
                best_combo = combo

        if best_combo:
            ref_slice_start = max(0.0, best_combo[0]['start'])
            ref_slice_end = min(orig_duration, best_combo[-1]['end'])

    extract_audio_slice(full_wav, ref_slice_start, ref_slice_end, output_slice_wav)

    ref_text = ""
    if whisper_model is not None:
        try:
            asr = whisper_model.transcribe(output_slice_wav)
            ref_text = asr.get('text', '').strip()
        except Exception:
            pass

    return ref_slice_start, ref_slice_end, ref_text

def translate_text_robust(text, target_code, max_chunk=2000):
    """
    Traduz textos com segurança, suporte a múltiplos idiomas e fallbacks automáticos.
    Evita erros 500 ou travamentos por limitação de requisições.
    """
    if not text or not text.strip():
        return ''
    clean_text = text.strip()

    api_target = target_code.lower()
    if api_target in ('pt-br', 'pt_br'):
        api_target = 'pt'
    elif api_target in ('zh-cn', 'zh_cn'):
        api_target = 'zh-CN'

    try:
        translated = GoogleTranslator(source='auto', target=api_target).translate(clean_text)
        if translated and translated.strip():
            return translated.strip()
    except Exception as e:
        print(f'Aviso: Tradução direta falhou ({e}), tentando em blocos...')

    sentences = re.split(r'(?<=[.!?;
])\s+', clean_text)
    chunks = []
    current = []
    curr_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if curr_len + len(s) + 1 > max_chunk:
            if current:
                chunks.append(' '.join(current))
            current = [s]
            curr_len = len(s)
        else:
            current.append(s)
            curr_len += len(s) + 1
    if current:
        chunks.append(' '.join(current))

    translated_parts = []
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue
        part = None
        for attempt in range(2):
            try:
                part = GoogleTranslator(source='auto', target=api_target).translate(chunk)
                time.sleep(0.1)
                break
            except Exception:
                time.sleep(0.3)
        if not part:
            part = chunk
        translated_parts.append(part)

    return ' '.join(translated_parts)

def build_atempo_filter(speed_factor):
    """Gera encadeamento de filtros atempo no FFmpeg (cada filtro suporta entre 0.5 e 2.0)."""
    speed = speed_factor
    filters = []
    while speed > 2.0:
        filters.append('atempo=2.0')
        speed /= 2.0
    while speed < 0.5:
        filters.append('atempo=0.5')
        speed /= 0.5
    filters.append(f'atempo={speed:.5f}')
    return ','.join(filters)

def time_sync_audio(synth_wav_path, target_duration, output_synced_wav, max_stretch=0.15):
    """
    Ajusta a duração da fala gerada preservando a qualidade acústica natural.
    Limita o time-stretch a no máximo ±15% para evitar vozes robóticas,
    e preenche o restante com padding natural.
    """
    synth_duration, _ = get_media_info(synth_wav_path)
    if synth_duration <= 0 or target_duration <= 0:
        shutil.copyfile(synth_wav_path, output_synced_wav)
        return 1.0, synth_duration

    raw_speed_factor = synth_duration / target_duration
    speed_factor = max(1.0 - max_stretch, min(1.0 + max_stretch, raw_speed_factor))

    if 0.98 <= speed_factor <= 1.02:
        filter_chain = f'apad=whole_dur={target_duration:.4f}'
    else:
        tempo_filter = build_atempo_filter(speed_factor)
        filter_chain = f'{tempo_filter},apad=whole_dur={target_duration:.4f}'

    cmd = [
        'ffmpeg', '-y', '-i', synth_wav_path,
        '-filter:a', filter_chain,
        '-t', f'{target_duration:.4f}',
        '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_synced_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return raw_speed_factor, target_duration

def assemble_timeline_audio(
    original_audio_path,
    synth_speech_wav_path,
    speech_start,
    speech_end,
    total_duration,
    output_final_wav,
    sync_duration=True
):
    """
    Monta a linha do tempo inteligente com crossfade suave:
    1. Preserva música/efeitos da introdução e encerramento originais.
    2. Encaixa a fala dublada na janela exata.
    3. Retorna duração idêntica à do vídeo original.
    """
    synth_dur, _ = get_media_info(synth_speech_wav_path)
    target_speech_dur = max(0.5, speech_end - speech_start)
    has_intro = speech_start >= 0.25
    has_outro = (total_duration - speech_end) >= 0.25

    temp_synced_speech = tempfile.NamedTemporaryFile(suffix='_synced_speech.wav', delete=False).name
    if sync_duration and target_speech_dur > 0:
        raw_speed, _ = time_sync_audio(synth_speech_wav_path, target_speech_dur, temp_synced_speech)
    else:
        shutil.copyfile(synth_speech_wav_path, temp_synced_speech)
        raw_speed = 1.0

    audio_parts = []

    if has_intro:
        temp_intro = tempfile.NamedTemporaryFile(suffix='_intro.wav', delete=False).name
        cmd_intro = [
            'ffmpeg', '-y',
            '-ss', '0.000',
            '-t', f'{speech_start:.3f}',
            '-i', original_audio_path,
            '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
            temp_intro
        ]
        subprocess.run(cmd_intro, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        t_intro, _ = torchaudio.load(temp_intro)
        audio_parts.append(t_intro)

    t_speech, _ = torchaudio.load(temp_synced_speech)
    t_speech = normalize_audio_tensor(t_speech, target_peak=0.95)
    audio_parts.append(t_speech)

    if has_outro and sync_duration:
        outro_dur = max(0.1, total_duration - speech_end)
        temp_outro = tempfile.NamedTemporaryFile(suffix='_outro.wav', delete=False).name
        cmd_outro = [
            'ffmpeg', '-y',
            '-ss', f'{speech_end:.3f}',
            '-t', f'{outro_dur:.3f}',
            '-i', original_audio_path,
            '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
            temp_outro
        ]
        subprocess.run(cmd_outro, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        t_outro, _ = torchaudio.load(temp_outro)
        audio_parts.append(t_outro)

    t_final = torch.cat(audio_parts, dim=-1)
    temp_cat = tempfile.NamedTemporaryFile(suffix='_cat.wav', delete=False).name
    torchaudio.save(temp_cat, t_final, 24000)

    if sync_duration and total_duration > 0:
        cmd_fit = [
            'ffmpeg', '-y', '-i', temp_cat,
            '-filter:a', f'apad=whole_dur={total_duration:.4f}',
            '-t', f'{total_duration:.4f}',
            '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
            output_final_wav
        ]
        subprocess.run(cmd_fit, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    else:
        shutil.copyfile(temp_cat, output_final_wav)

    final_dur, _ = get_media_info(output_final_wav)
    return raw_speed, final_dur

def remux_video_with_audio(original_video_path, new_audio_path, output_video_path):
    """
    Substitui a faixa de áudio do vídeo original pelo áudio dublado e sincronizado.
    Utiliza stream copy (-c:v copy) para renderização instantânea sem perda de qualidade visual.
    """
    cmd = [
        'ffmpeg', '-y',
        '-i', original_video_path,
        '-i', new_audio_path,
        '-map', '0:v:0',
        '-map', '1:a:0',
        '-c:v', 'copy',
        '-c:a', 'aac',
        '-b:a', '192k',
        '-shortest',
        output_video_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def generate_voice_cloning_chunked(model, text, ref_audio, ref_text, num_steps=32, speed=1.0, max_chunk_chars=450, progress_callback=None):
    """
    Gera fala com OmniVoice com chunking inteligente por frases naturais,
    normalização de áudio e pausas naturais entre sentenças.
    """
    if len(text) <= max_chunk_chars:
        if progress_callback:
            progress_callback(0, 1, text)
        out = model.generate(
            text=text,
            ref_audio=ref_audio,
            ref_text=ref_text if ref_text else None,
            num_step=int(num_steps),
            speed=float(speed)
        )
        t = out[0] if isinstance(out[0], torch.Tensor) else torch.tensor(out[0])
        t = t.unsqueeze(0) if t.dim() == 1 else t
        return normalize_audio_tensor(t)

    sentences = re.split(r'(?<=[.!?;
])\s+', text)
    chunks = []
    current = []
    curr_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if curr_len + len(s) + 1 > max_chunk_chars:
            if current:
                chunks.append(' '.join(current))
            current = [s]
            curr_len = len(s)
        else:
            current.append(s)
            curr_len += len(s) + 1
    if current:
        chunks.append(' '.join(current))

    tensors = []
    silence = torch.zeros((1, int(24000 * 0.12)))

    for idx, c in enumerate(chunks):
        if progress_callback:
            progress_callback(idx, len(chunks), c)
        out = model.generate(
            text=c,
            ref_audio=ref_audio,
            ref_text=ref_text if ref_text else None,
            num_step=int(num_steps),
            speed=float(speed)
        )
        t = out[0] if isinstance(out[0], torch.Tensor) else torch.tensor(out[0])
        if t.dim() == 1:
            t = t.unsqueeze(0)
        t = normalize_audio_tensor(t)
        tensors.append(t)
        tensors.append(silence)

    if tensors:
        tensors.pop()
        combined = torch.cat(tensors, dim=-1)
        return normalize_audio_tensor(combined)
    return torch.zeros((1, 24000))

print('✅ Motor de processamento e sincronização com áudio de alta fidelidade carregado!')


In [ ]:
# @title Passo 4: Iniciar a Interface de Dublagem Sincronizada (Gradio)
# @markdown Clique no botão 'Play' e acesse o link público 'Running on public URL: https://...gradio.live'

import os
import re
import shutil
import tempfile
import gc
import time
import torch
import torchaudio
import gradio as gr

# Diretório dedicado e seguro para saídas (resolve permissões e erro 500 no Gradio)
OUTPUTS_DIR = os.path.abspath("./outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Idiomas suportados com destaque para Português Brasileiro e idiomas globais
LANGUAGES = {
    '🇧🇷 Português Brasileiro (pt-BR)': 'pt-BR',
    '🇺🇸 Inglês (English)': 'en',
    '🇪🇸 Espanhol (Español)': 'es',
    '🇫🇷 Francês (Français)': 'fr',
    '🇩🇪 Alemão (Deutsch)': 'de',
    '🇨🇳 Chinês Simplificado (中文)': 'zh-CN',
    '🇸🇦 Árabe (العربية)': 'ar',
    '🇮🇹 Italiano (Italiano)': 'it',
    '🇯🇵 Japonês (日本語)': 'ja',
    '🇷🇺 Russo (Русский)': 'ru'
}

def resolve_lang_code(lang_label):
    if lang_label in LANGUAGES:
        return LANGUAGES[lang_label]
    l_lower = lang_label.lower()
    if 'espanh' in l_lower or 'es' in l_lower:
        return 'es'
    if 'ingl' in l_lower or 'en' in l_lower:
        return 'en'
    if 'portug' in l_lower or 'pt' in l_lower:
        return 'pt-BR'
    if 'franc' in l_lower or 'fr' in l_lower:
        return 'fr'
    if 'alem' in l_lower or 'de' in l_lower:
        return 'de'
    if 'chin' in l_lower or 'zh' in l_lower:
        return 'zh-CN'
    if 'arab' in l_lower or 'ar' in l_lower:
        return 'ar'
    if 'ital' in l_lower or 'it' in l_lower:
        return 'it'
    if 'japon' in l_lower or 'ja' in l_lower:
        return 'ja'
    if 'russ' in l_lower or 'ru' in l_lower:
        return 'ru'
    return 'en'

def transcribe_only(media_file, progress=gr.Progress()):
    if not media_file:
        return '', '⚠️ Por favor, envie um arquivo de vídeo ou áudio antes de transcrever.'
    try:
        if progress is not None:
            progress(0.15, desc='Extraindo faixa de áudio...')

        orig_dur, _ = get_media_info(media_file)
        temp_wav = tempfile.NamedTemporaryFile(suffix='_transcribe.wav', delete=False).name
        extract_audio_to_wav(media_file, temp_wav)

        if progress is not None:
            progress(0.50, desc='Transcrevendo fala com Whisper...')
        asr_res = whisper_model.transcribe(temp_wav)
        text = asr_res.get('text', '').strip()
        lang = asr_res.get('language', 'desconhecido')
        segments = asr_res.get('segments', [])

        speech_start = float(segments[0]['start']) if segments else 0.0

        if progress is not None:
            progress(1.0, desc='Transcrição concluída!')

        intro_text = f" | 🎵 **Intro detectada:** `{speech_start:.2f}s`" if speech_start >= 0.25 else ""
        status_msg = f"""✅ **Transcrição concluída com sucesso!**
- ⏱️ **Duração da Mídia:** `{orig_dur:.2f}s` | 🌐 **Idioma detectado:** `{lang.upper()}`{intro_text}
> 💡 *Você pode ler e editar qualquer palavra no campo abaixo. Em seguida, selecione os idiomas e clique em **'2. Dublar e Sincronizar'**.*"""
        return text, status_msg
    except Exception as e:
        return '', f'❌ **Erro ao transcrever áudio:** `{str(e)}`'

def process_dubbing(media_file, edited_transcription, target_lang_labels, custom_ref_audio, sync_duration_opt, num_steps, user_speed, progress=gr.Progress()):
    if not media_file:
        return (
            None, None, [],
            '❌ **Erro:** Por favor, envie um arquivo de vídeo (.mp4, .mov, etc.) ou de áudio (.wav, .mp3, etc.).',
            '', ''
        )

    if not target_lang_labels or len(target_lang_labels) == 0:
        return (
            None, None, [],
            '❌ **Erro:** Por favor, marque pelo menos um idioma de destino nas caixas de seleção.',
            '', ''
        )

    try:
        if progress is not None:
            progress(0.05, desc='Analisando arquivo de mídia...')

        orig_duration, has_video = get_media_info(media_file)

        temp_full_wav = tempfile.NamedTemporaryFile(suffix='_full.wav', delete=False).name
        extract_audio_to_wav(media_file, temp_full_wav)

        if progress is not None:
            progress(0.12, desc='Analisando voz original com Whisper...')
        asr_result = whisper_model.transcribe(temp_full_wav)
        whisper_text = asr_result.get('text', '').strip()
        detected_lang = asr_result.get('language', 'desconhecido')
        segments = asr_result.get('segments', [])

        original_text = (edited_transcription or '').strip()
        if not original_text:
            original_text = whisper_text

        if not original_text:
            return (
                None, None, [],
                '❌ **Erro:** Não foi possível reconhecer fala audível no arquivo enviado.',
                '', ''
            )

        speech_start = float(segments[0]['start']) if segments else 0.0
        speech_end = float(segments[-1]['end']) if segments else orig_duration
        target_speech_dur = max(0.5, speech_end - speech_start)

        # 3. Referência vocal de alta fidelidade
        ref_slice_wav = tempfile.NamedTemporaryFile(suffix='_ref_slice.wav', delete=False).name
        if custom_ref_audio and os.path.exists(custom_ref_audio):
            print('🎙️ Utilizando áudio de referência limpo enviado pelo usuário!')
            extract_audio_to_wav(custom_ref_audio, ref_slice_wav)
            ref_asr = whisper_model.transcribe(ref_slice_wav)
            ref_slice_text = ref_asr.get('text', '').strip()
        else:
            print('🎙️ Selecionando trecho vocal mais limpo da mídia original...')
            _, _, ref_slice_text = select_best_voice_slice(
                segments=segments,
                full_wav=temp_full_wav,
                orig_duration=orig_duration,
                output_slice_wav=ref_slice_wav,
                whisper_model=whisper_model
            )

        orig_base_name = os.path.splitext(os.path.basename(media_file))[0]
        clean_base_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', orig_base_name)

        generated_files = []
        translations_summary = []
        results_info = []

        first_preview_audio = None
        first_preview_video = None

        total_langs = len(target_lang_labels)
        print(f'🚀 Iniciando dublagem para {total_langs} idioma(s): {target_lang_labels}')

        for idx, lang_label in enumerate(target_lang_labels):
            lang_code = resolve_lang_code(lang_label)
            clean_code = lang_code.replace('-', '_')
            step_base = 0.20 + (0.75 * idx / total_langs)
            lang_span = 0.75 / total_langs

            if progress is not None:
                progress(step_base, desc=f'[{idx+1}/{total_langs}] Traduzindo para {lang_label}...')

            translated_text = translate_text_robust(original_text, lang_code)
            translations_summary.append(f'### {lang_label}\n{translated_text}\n')

            cloning_start = step_base + (lang_span * 0.15)
            cloning_span = lang_span * 0.60

            def chunk_progress(chunk_idx, num_chunks, chunk_txt):
                if progress is not None:
                    p = cloning_start + (cloning_span * (chunk_idx + 1) / max(1, num_chunks))
                    progress(p, desc=f'[{idx+1}/{total_langs}] {lang_label}: Frase {chunk_idx+1}/{num_chunks}...')

            if progress is not None:
                progress(cloning_start, desc=f'[{idx+1}/{total_langs}] Clonando voz em {lang_label}...')

            audio_tensor = generate_voice_cloning_chunked(
                model=omnivoice_model,
                text=translated_text,
                ref_audio=ref_slice_wav,
                ref_text=ref_slice_text,
                num_steps=int(num_steps),
                speed=float(user_speed),
                progress_callback=chunk_progress
            )

            temp_synth_wav = tempfile.NamedTemporaryFile(suffix='_synth.wav', delete=False).name
            torchaudio.save(temp_synth_wav, audio_tensor.cpu(), 24000)

            if progress is not None:
                progress(step_base + (lang_span * 0.80), desc=f'[{idx+1}/{total_langs}] Sincronizando áudio...')

            final_audio_path = os.path.join(OUTPUTS_DIR, f'{clean_base_name}_{clean_code}_audio.wav')
            speed_factor, final_duration = assemble_timeline_audio(
                original_audio_path=temp_full_wav,
                synth_speech_wav_path=temp_synth_wav,
                speech_start=speech_start,
                speech_end=speech_end,
                total_duration=orig_duration,
                output_final_wav=final_audio_path,
                sync_duration=sync_duration_opt
            )
            generated_files.append(final_audio_path)
            if first_preview_audio is None:
                first_preview_audio = final_audio_path

            final_video_path = None
            if has_video:
                if progress is not None:
                    progress(step_base + (lang_span * 0.92), desc=f'[{idx+1}/{total_langs}] Renderizando vídeo MP4...')
                final_video_path = os.path.join(OUTPUTS_DIR, f'{clean_base_name}_{clean_code}_dublado.mp4')
                remux_video_with_audio(media_file, final_audio_path, final_video_path)
                generated_files.append(final_video_path)
                if first_preview_video is None:
                    first_preview_video = final_video_path

            results_info.append({
                'label': lang_label,
                'code': lang_code,
                'final_dur': final_duration,
                'speed_factor': speed_factor,
                'audio_file': os.path.basename(final_audio_path),
                'video_file': os.path.basename(final_video_path) if final_video_path else None
            })

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

        if progress is not None:
            progress(1.0, desc='Processamento concluído com sucesso!')

        rows_md = []
        for r in results_info:
            vid_col = f"`{r['video_file']}`" if r['video_file'] else 'N/A'
            tempo_col = f"{r['speed_factor']:.2f}x" if sync_duration_opt else 'Original'
            rows_md.append(f"| **{r['label']}** | `{r['audio_file']}` | {vid_col} | `{tempo_col}` | `{r['final_dur']:.2f}s` |")
        table_body = '\n'.join(rows_md)

        intro_desc = f"- 🎵 **Música de Intro Preservada:** até `{speech_start:.2f}s`\n" if speech_start >= 0.25 else ""
        outro_desc = f"- 🎵 **Música de Encerramento Preservada:** a partir de `{speech_end:.2f}s`\n" if (orig_duration - speech_end) >= 0.25 else ""

        status_md = f"""### ✅ Dublagem Concluída com Sucesso! ({len(results_info)} idioma(s) gerado(s))
- 🌐 **Idioma Original Detectado:** `{detected_lang.upper()}`
- ⏱️ **Duração Total:** `{orig_duration:.2f}s` (idêntica ao original)
{intro_desc}{outro_desc}- 🎙️ **Janela da Fala:** `{speech_start:.2f}s` até `{speech_end:.2f}s` (`{target_speech_dur:.2f}s`)
- 📁 **Arquivos Salvos:** Disponíveis para pré-visualização e download abaixo.

| Idioma | Áudio WAV | Vídeo MP4 Dublado | Ajuste | Duração Final |
| :--- | :--- | :--- | :--- | :--- |
{table_body}
"""
        all_translations = '\n\n---\n\n'.join(translations_summary)
        return first_preview_audio, first_preview_video, generated_files, status_md, original_text, all_translations

    except Exception as e:
        import traceback
        traceback.print_exc()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return (
            None, None, [],
            f'❌ **Erro durante a execução:** `{str(e)}`',
            edited_transcription or '', ''
        )

def process_free_tts(custom_text, ref_audio, num_steps, speed):
    if not custom_text or not custom_text.strip():
        return None, '⚠️ Digite um texto para sintetizar.'
    if not ref_audio:
        return None, '⚠️ Envie uma amostra de áudio com a voz a ser clonada.'
    try:
        ref_wav = tempfile.NamedTemporaryFile(suffix='_free_ref.wav', delete=False).name
        extract_audio_to_wav(ref_audio, ref_wav)
        ref_asr = whisper_model.transcribe(ref_wav)
        ref_text = ref_asr.get('text', '').strip()

        audio_tensor = generate_voice_cloning_chunked(
            model=omnivoice_model,
            text=custom_text.strip(),
            ref_audio=ref_wav,
            ref_text=ref_text,
            num_steps=int(num_steps),
            speed=float(speed)
        )
        out_name = f"clonagem_livre_{int(time.time())}.wav"
        final_path = os.path.join(OUTPUTS_DIR, out_name)
        torchaudio.save(final_path, audio_tensor.cpu(), 24000)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return final_path, '✅ Fala sintetizada com sucesso com a sua voz!'
    except Exception as e:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        return None, f'❌ Erro: {str(e)}'

with gr.Blocks(title='Matraca Studio - Dublador & Clonador de Voz com IA', theme=gr.themes.Soft(primary_hue='emerald', secondary_hue='blue')) as demo:
    gr.HTML("""
    <div style='text-align: center; margin-bottom: 18px;'>
        <h1 style='font-size: clamp(1.2rem, 3.2vw, 2.0rem); font-weight: 700; margin-bottom: 6px;'>🎙️ Matraca Studio - Dublador & Clonador de Voz com IA</h1>
        <p style='color: #555; font-size: 1.1em;'>
            Clone sua voz e duble qualquer vídeo MP4 ou áudio para <b>múltiplos idiomas</b>
            mantendo <b>alta fidelidade acústica</b> e o <b>tempo original</b> do vídeo!
        </p>
    </div>
    """)

    with gr.Tabs():
        # --- ABA 1: DUBLAGEM SINCRONIZADA ---
        with gr.TabItem('🎬 Dublagem Sincronizada (Vídeo ou Áudio)'):
            with gr.Row():
                with gr.Column(scale=1):
                    input_media = gr.File(
                        label='1. Envie seu Vídeo (MP4, MOV, MKV) ou Áudio (WAV, MP3, M4A)',
                        file_count='single',
                        type='filepath'
                    )

                    with gr.Accordion('🎙️ Referência Vocal Personalizada (Opcional - Recomendado se o vídeo tiver música alta)', open=False):
                        custom_ref_voice = gr.Audio(
                            label='Envie 5 a 10 segundos da sua voz limpa gravada no microfone/celular',
                            type='filepath'
                        )

                    btn_transcribe = gr.Button('🔍 1. Transcrever e Analisar Áudio Original', variant='secondary', size='md')
                    status_transcribe = gr.Markdown('')

                    txt_orig = gr.Textbox(
                        label='Transcrição do Áudio Original (Revise e edite as palavras se desejar):',
                        placeholder='Clique em "1. Transcrever e Analisar Áudio Original" acima para transcrever, ou digite/cole o texto aqui...',
                        lines=4,
                        interactive=True
                    )

                    target_langs = gr.CheckboxGroup(
                        choices=list(LANGUAGES.keys()),
                        value=['🇺🇸 Inglês (English)', '🇪🇸 Espanhol (Español)'],
                        label='2. Idiomas de Destino da Dublagem (Selecione um ou vários)',
                        info='Marque os idiomas desejados. Cada um será processado sequencialmente de forma otimizada.'
                    )
                    with gr.Row():
                        btn_select_all = gr.Button('➕ Selecionar Todos', size='sm')
                        btn_clear_all = gr.Button('✖️ Limpar Seleção', size='sm')

                    sync_checkbox = gr.Checkbox(
                        value=True,
                        label='⏱️ Sincronizar Linha do Tempo e Duração (Preserva música de intro/encerramento)',
                        info='Preserva a música de introdução/encerramento original e encaixa a fala dublada perfeitamente.'
                    )
                    with gr.Accordion('⚙️ Configurações Avançadas de IA', open=False):
                        steps_slider = gr.Slider(minimum=16, maximum=64, value=32, step=4, label='Passos de Difusão (Diffusion Steps - 32 recomendado)')
                        speed_slider = gr.Slider(minimum=0.8, maximum=1.3, value=1.0, step=0.05, label='Velocidade Base da Fala')

                    btn_dub = gr.Button('🚀 2. Dublar e Sincronizar Vídeo/Áudio', variant='primary', size='lg')

                with gr.Column(scale=1):
                    status_label = gr.Markdown('')

                    with gr.Row():
                        audio_preview = gr.Audio(label='🔊 Prévia do Áudio Dublado', type='filepath', interactive=False)
                        video_preview = gr.Video(label='🎬 Prévia do Vídeo Dublado', interactive=False)

                    output_files = gr.File(
                        label='📦 Download de Todos os Arquivos Gerados (WAV / MP4)',
                        file_count='multiple',
                        type='filepath',
                        interactive=False
                    )
                    with gr.Accordion('📝 Traduções Geradas por Idioma', open=True):
                        txt_trans = gr.Textbox(label='Traduções Geradas por Idioma', lines=6, interactive=False)

            btn_select_all.click(fn=lambda: list(LANGUAGES.keys()), outputs=[target_langs])
            btn_clear_all.click(fn=lambda: [], outputs=[target_langs])

            btn_transcribe.click(
                fn=transcribe_only,
                inputs=[input_media],
                outputs=[txt_orig, status_transcribe]
            )

            btn_dub.click(
                fn=process_dubbing,
                inputs=[input_media, txt_orig, target_langs, custom_ref_voice, sync_checkbox, steps_slider, speed_slider],
                outputs=[audio_preview, video_preview, output_files, status_label, txt_orig, txt_trans]
            )

        # --- ABA 2: CLONAGEM LIVRE ---
        with gr.TabItem('🎤 Clonagem Livre (Digitar Texto Personalizado)'):
            gr.Markdown('Envie uma amostra de áudio com a sua voz e digite qualquer texto em qualquer idioma para sintetizar diretamente:')
            with gr.Row():
                with gr.Column(scale=1):
                    ref_audio_free = gr.Audio(label='Áudio de Referência (sua voz)', type='filepath')
                    custom_text = gr.Textbox(
                        label='Texto a ser falado',
                        placeholder='Ex: Olá pessoal! Hoje estamos demonstrando a nova tecnologia de clonagem e dublagem com inteligência artificial.',
                        lines=4
                    )
                    with gr.Row():
                        steps_free = gr.Slider(minimum=16, maximum=64, value=32, step=4, label='Passos de Difusão')
                        speed_free = gr.Slider(minimum=0.7, maximum=1.4, value=1.0, step=0.05, label='Velocidade')
                    btn_free = gr.Button('Gerar Áudio com Minha Voz', variant='primary', size='lg')
                    status_free = gr.Markdown('')
                with gr.Column(scale=1):
                    audio_free_out = gr.Audio(label='Áudio Sintetizado', type='filepath', interactive=False)

            btn_free.click(
                fn=process_free_tts,
                inputs=[custom_text, ref_audio_free, steps_free, speed_free],
                outputs=[audio_free_out, status_free]
            )

# Habilita a fila (Queue) - Essencial para evitar Erro 500 / Timeout e suportar barras de progresso
demo.queue(max_size=32, default_concurrency_limit=1).launch(
    share=True,
    debug=True,
    allowed_paths=[OUTPUTS_DIR]
)
